# 🛢️ Proyecto Integrador — Modelado Predictivo de Producción Petrolera
**Fuente de datos:** Google Sheets (actualización automática)

---

In [ ]:
# ================================================================
# CELDA 0 — CONFIGURACIÓN GLOBAL (Correr primero siempre)
# ================================================================
import io
import os
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# ----------------------------------------------------------------
# ARCHIVO CSV LOCAL — Dataset descargado desde AWS
# ----------------------------------------------------------------
ARCHIVO_CSV = 'produccion_limpia.csv'

print('📦 Cargando datos desde archivo local...')
df_base = pd.read_csv(ARCHIVO_CSV)
df_base['Date'] = pd.to_datetime(df_base['Date'])
df_base = df_base.sort_values(by=['Well_ID', 'Date']).reset_index(drop=True)

print(f'✅ Datos cargados: {df_base.shape[0]} registros | {df_base["Well_ID"].nunique()} pozos')
print(f'📋 Columnas disponibles: {list(df_base.columns)}')
df_base.head()

In [ ]:
# ================================================================
# CELDA 0.5 — DISTRIBUCIONES DE VARIABLES NUMÉRICAS
# ================================================================
import math
import scipy.stats as stats
import seaborn as sns

print("INTERPRETACIÓN DE LAS DISTRIBUCIONES DE LAS VARIABLES NUMÉRICAS")

# Seleccionar solo columnas numéricas de df_base
variables_numericas = df_base.select_dtypes(include=[np.number])
print("\nInformación de variables numéricas:")
variables_numericas.info()

# =====================
# HISTOGRAMAS
# =====================
plt.figure(figsize=(15, 10))
variables_numericas.hist(bins=30, figsize=(12, 12))
plt.suptitle("HISTOGRAMAS VARIABLES NUMÉRICAS")
plt.tight_layout()
plt.show()

# =====================
# DIAGRAMAS DE BIGOTES
# =====================
numCols = len(variables_numericas.columns)
cols = 4
rows = math.ceil(numCols / cols)

plt.figure(figsize=(14, 10))
plt.suptitle("DIAGRAMAS DE BIGOTES")
for i, column in enumerate(variables_numericas.columns, 1):
    plt.subplot(rows, cols, i)
    sns.boxplot(x=variables_numericas[column], color='red')
    plt.title(column)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# =====================
# QQ PLOTS
# =====================
numCols = len(variables_numericas.columns)
cols = 4
rows = math.ceil(numCols / cols)

plt.figure(figsize=(14, 12))
plt.suptitle("QQ PLOTS VARIABLES NUMÉRICAS")
for i, column in enumerate(variables_numericas.columns, 1):
    plt.subplot(rows, cols, i)
    stats.probplot(variables_numericas[column].dropna(), dist='norm', plot=plt)
    plt.title(column)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

---
## 📊 Visualizaciones Exploratorias

In [ ]:
# ================================================================
# CELDA 1 — VOLÚMENES ACUMULADOS (CumOil + CumWater)
# ================================================================
def analizar_acumulados_pozos(df):
    df = df.copy()
    pozos = df['Well_ID'].unique()
    print(f'🔎 Analizando volúmenes acumulados para {len(pozos)} pozos')

    plt.rcParams['axes.grid'] = True
    plt.rcParams['grid.alpha'] = 0.3
    plt.rcParams['grid.linestyle'] = '--'

    for well in pozos:
        df_well = df[df['Well_ID'] == well].copy()
        df_well['Lift_Changed'] = (df_well['Lift_system'] != df_well['Lift_system'].shift(1))
        df_well.iloc[0, df_well.columns.get_loc('Lift_Changed')] = False
        fechas_cambio = df_well[df_well['Lift_Changed'] == True]

        fig, ax1 = plt.subplots(figsize=(14, 7))
        ax2 = ax1.twinx()

        (line1,) = ax1.plot(df_well['Date'], df_well['CumOil_bbl'], color='#2ca02c', linewidth=2.5, label='Petróleo Acumulado (CumOil)')
        ax1.set_ylabel('Petróleo Acumulado (bbl)', color='#2ca02c', fontsize=12)
        ax1.tick_params(axis='y', labelcolor='#2ca02c')

        (line2,) = ax2.plot(df_well['Date'], df_well['CumWater_bbl'], color='#1f77b4', linewidth=2.5, linestyle='-.', label='Agua Acumulada (CumWater)')
        ax2.set_ylabel('Agua Acumulada (bbl)', color='#1f77b4', fontsize=12)
        ax2.tick_params(axis='y', labelcolor='#1f77b4')

        for idx, row in fechas_cambio.iterrows():
            ax1.axvline(x=row['Date'], color='red', linestyle=':', linewidth=2, alpha=0.8)
            y_pos = df_well['CumOil_bbl'].max() * 0.5
            ax1.annotate(f"Cambio a:\n{row['Lift_system']}", xy=(row['Date'], y_pos),
                         xytext=(15, -20), textcoords='offset points',
                         arrowprops=dict(arrowstyle='->', color='red'), bbox=dict(boxstyle='round,pad=0.3', fc='yellow', alpha=0.6), fontsize=9, fontweight='bold')

        ticks_indices = np.linspace(0, len(df_well) - 1, 8, dtype=int)
        dates_sampled = df_well['Date'].iloc[ticks_indices]
        lift_sampled = df_well['Lift_system'].iloc[ticks_indices]
        x_labels = [f"{d.strftime('%Y-%m-%d')}\n({l})" for d, l in zip(dates_sampled, lift_sampled)]
        ax1.set_xticks(dates_sampled)
        ax1.set_xticklabels(x_labels, rotation=15, ha='right', fontsize=9)

        plt.title(f'Análisis de Volúmenes Acumulados — Pozo: {well}', fontsize=14, fontweight='bold', pad=15)
        ax1.legend([line1, line2], [line1.get_label(), line2.get_label()], loc='upper left')
        plt.tight_layout()
        plt.show()

analizar_acumulados_pozos(df_base)

In [ ]:
# ================================================================
# CELDA 2 — TASAS DIARIAS (Oil + Water)
# ================================================================
def analizar_tasas_diarias(df):
    df = df.copy()
    pozos = df['Well_ID'].unique()
    print(f'🔎 Analizando tasas diarias para {len(pozos)} pozos')

    for well in pozos:
        df_well = df[df['Well_ID'] == well].copy()
        df_well['Lift_Changed'] = (df_well['Lift_system'] != df_well['Lift_system'].shift(1))
        df_well.iloc[0, df_well.columns.get_loc('Lift_Changed')] = False
        fechas_cambio = df_well[df_well['Lift_Changed'] == True]

        fig, ax1 = plt.subplots(figsize=(14, 7))
        ax2 = ax1.twinx()

        (line1,) = ax1.plot(df_well['Date'], df_well['Oil_rate_bbl_d'], color='#2ca02c', linewidth=2, label='Tasa de Petróleo (Oil Rate)')
        ax1.set_ylabel('Oil Rate (bbl/d)', color='#2ca02c', fontsize=12)
        ax1.tick_params(axis='y', labelcolor='#2ca02c')

        (line2,) = ax2.plot(df_well['Date'], df_well['Water_rate_bbl_d'], color='#1f77b4', linewidth=1.5, alpha=0.8, label='Tasa de Agua (Water Rate)')
        ax2.set_ylabel('Water Rate (bbl/d)', color='#1f77b4', fontsize=12)
        ax2.tick_params(axis='y', labelcolor='#1f77b4')

        for idx, row in fechas_cambio.iterrows():
            ax1.axvline(x=row['Date'], color='red', linestyle=':', linewidth=2, alpha=0.8)
            y_pos = df_well['Oil_rate_bbl_d'].max() * 0.8
            ax1.annotate(f"Cambio a:\n{row['Lift_system']}", xy=(row['Date'], y_pos),
                         xytext=(15, 15), textcoords='offset points',
                         arrowprops=dict(arrowstyle='->', color='red'), bbox=dict(boxstyle='round,pad=0.3', fc='yellow', alpha=0.6), fontsize=9, fontweight='bold')

        ticks_indices = np.linspace(0, len(df_well) - 1, 8, dtype=int)
        dates_sampled = df_well['Date'].iloc[ticks_indices]
        lift_sampled = df_well['Lift_system'].iloc[ticks_indices]
        x_labels = [f"{d.strftime('%Y-%m-%d')}\n({l})" for d, l in zip(dates_sampled, lift_sampled)]
        ax1.set_xticks(dates_sampled)
        ax1.set_xticklabels(x_labels, rotation=15, ha='right', fontsize=9)

        plt.title(f'Análisis de Tasas Diarias — Pozo: {well}', fontsize=14, fontweight='bold', pad=15)
        ax1.legend([line1, line2], [line1.get_label(), line2.get_label()], loc='upper left')
        plt.tight_layout()
        plt.show()

analizar_tasas_diarias(df_base)

In [ ]:
# ================================================================
# CELDA 3 — DIAGNÓSTICO TRIVARIABLE (Oil + Water + Gas)
# ================================================================
def analizar_comportamiento_trivariable(df):
    df = df.copy()
    if 'Well_Type' in df.columns:
        df = df[df['Well_Type'].str.lower() == 'producer'].copy()

    pozos = df['Well_ID'].unique()
    print(f'🔎 Generando gráficos trivariables para {len(pozos)} pozos productores')

    plt.rcParams['axes.grid'] = True
    plt.rcParams['grid.alpha'] = 0.2
    plt.rcParams['grid.linestyle'] = '--'

    for well in pozos:
        df_well = df[df['Well_ID'] == well].copy()

        if 'Lift_change_flag' in df_well.columns:
            fechas_cambio = df_well[df_well['Lift_change_flag'] == 1]
        else:
            df_well['Lift_Changed'] = (df_well['Lift_system'] != df_well['Lift_system'].shift(1))
            df_well.iloc[0, df_well.columns.get_loc('Lift_Changed')] = False
            fechas_cambio = df_well[df_well['Lift_Changed'] == True]

        fig, ax1 = plt.subplots(figsize=(15, 8))
        ax2 = ax1.twinx()
        ax3 = ax1.twinx()
        ax3.spines['right'].set_position(('axes', 1.08))

        (line1,) = ax1.plot(df_well['Date'], df_well['Oil_rate_bbl_d'], color='#2ca02c', linewidth=2.5, label='Oil Rate (bbl/d)')
        ax1.set_ylabel('Petróleo: Oil Rate (bbl/d)', color='#2ca02c', fontsize=11)
        ax1.tick_params(axis='y', labelcolor='#2ca02c')

        (line2,) = ax2.plot(df_well['Date'], df_well['Water_rate_bbl_d'], color='#1f77b4', linewidth=1.5, alpha=0.7, label='Water Rate (bbl/d)')
        ax2.set_ylabel('Agua: Water Rate (bbl/d)', color='#1f77b4', fontsize=11)
        ax2.tick_params(axis='y', labelcolor='#1f77b4')

        (line3,) = ax3.plot(df_well['Date'], df_well['Gas_rate_mscf_d'], color='#e377c2', linewidth=1.5, linestyle='-.', alpha=0.8, label='Gas Rate (mscf/d)')
        ax3.set_ylabel('Gas: Gas Rate (mscf/d)', color='#e377c2', fontsize=11)
        ax3.tick_params(axis='y', labelcolor='#e377c2')

        for idx, row in fechas_cambio.iterrows():
            ax1.axvline(x=row['Date'], color='red', linestyle=':', linewidth=2, alpha=0.9)
            y_pos = df_well['Oil_rate_bbl_d'].max() * 0.85 if df_well['Oil_rate_bbl_d'].max() > 0 else 100
            ax1.annotate(f"Cambio a:\n{row['Lift_system']}", xy=(row['Date'], y_pos),
                         xytext=(10, 10), textcoords='offset points',
                         arrowprops=dict(arrowstyle='->', color='red'), bbox=dict(boxstyle='round,pad=0.3', fc='yellow', alpha=0.7), fontsize=8, fontweight='bold')

        ticks_indices = np.linspace(0, len(df_well) - 1, 8, dtype=int)
        dates_sampled = df_well['Date'].iloc[ticks_indices]
        lift_sampled = df_well['Lift_system'].iloc[ticks_indices]
        x_labels = [f"{d.strftime('%Y-%m-%d')}\n({l})" for d, l in zip(dates_sampled, lift_sampled)]
        ax1.set_xticks(dates_sampled)
        ax1.set_xticklabels(x_labels, rotation=15, ha='right', fontsize=9)

        plt.title(f'Diagnóstico Dinámico Multifásico — Pozo: {well}', fontsize=13, fontweight='bold', pad=15)
        ax1.legend([line1, line2, line3], [line1.get_label(), line2.get_label(), line3.get_label()], loc='upper left', frameon=True, shadow=True)
        plt.tight_layout()
        plt.show()

analizar_comportamiento_trivariable(df_base)

---
## 🧠 Modelado Predictivo

In [ ]:
# ================================================================
# CELDA 4 — PASO 1: CALCULAR TARGET (Dias_Restantes)
# Se genera en memoria — no requiere guardar ni cargar Excel
# ================================================================
def calcular_target_hibrido_produccion(df):
    print('⏳ Calculando Dias_Restantes con criterio híbrido (Petróleo + Agua)...')

    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by=['Well_ID', 'Date']).reset_index(drop=True)
    df = df[df['Well_Type'].str.lower() == 'producer'].copy()
    df['Dias_Restantes'] = np.nan

    pozos = df['Well_ID'].unique()

    for well in pozos:
        df_well = df[df['Well_ID'] == well].copy()
        indices_well = df_well.index

        df_well['Oil_7d_mean'] = df_well['Oil_rate_bbl_d'].rolling(window=7, min_periods=1).mean()
        df_well['Oil_prev_7d_mean'] = df_well['Oil_7d_mean'].shift(7)
        df_well['Oil_futuro_30d_mean'] = df_well['Oil_rate_bbl_d'].shift(-30).rolling(window=30, min_periods=1).mean()
        df_well['Water_7d_mean'] = df_well['Water_rate_bbl_d'].rolling(window=7, min_periods=1).mean()

        df_well['Colapso_Real'] = (
            (df_well['Oil_7d_mean'] < (df_well['Oil_prev_7d_mean'] * 0.75)) &
            (df_well['Oil_futuro_30d_mean'] < (df_well['Oil_prev_7d_mean'] * 0.80)) &
            (df_well['Water_7d_mean'] > 1500)
        )

        fechas_colapso = df_well[df_well['Colapso_Real'] == True]['Date'].tolist()

        if not fechas_colapso:
            fecha_limite = df_well['Date'].max()
        else:
            fecha_limite = fechas_colapso[0]
            print(f'🎯 Pozo {well}: Colapso verificado el {fecha_limite.strftime("%Y-%m-%d")}')

        df.loc[indices_well, 'Dias_Restantes'] = (fecha_limite - df_well['Date']).dt.days

    df['Dias_Restantes'] = df['Dias_Restantes'].apply(lambda x: max(0, x))
    print(f'\n✅ Columna Dias_Restantes generada en memoria — {df.shape[0]} registros listos')
    return df


# Encadenado directo desde df_base
df_modelo = calcular_target_hibrido_produccion(df_base)

# Eliminar filas con NaN en las features clave para el modelo
# Esto evita el error 'Input X contains NaN' en GradientBoosting y LSTM
# Se limpian solo las columnas que usa el modelo, sin afectar df_base original
features_modelo = ['Oil_rate_bbl_d', 'Water_rate_bbl_d', 'Gas_rate_mscf_d', 'Dias_Restantes']
antes = df_modelo.shape[0]
df_modelo = df_modelo.dropna(subset=features_modelo).reset_index(drop=True)
print(f'🧹 Limpieza: {antes - df_modelo.shape[0]} filas con NaN eliminadas | {df_modelo.shape[0]} registros válidos')

df_modelo[['Well_ID', 'Date', 'Oil_rate_bbl_d', 'Water_rate_bbl_d', 'Gas_rate_mscf_d', 'Dias_Restantes']].head(10)

In [ ]:
# ================================================================
# CELDA 5 — PASO 2: CREAR VENTANAS 3D PARA LA LSTM
# Recibe df_modelo directamente desde memoria
# ================================================================
def crear_ventanas_3d(df, tamano_ventana=30):
    print(f'📦 Estructurando tensores 3D con ventana de {tamano_ventana} días...')

    features = ['Oil_rate_bbl_d', 'Water_rate_bbl_d', 'Gas_rate_mscf_d']
    target = 'Dias_Restantes'

    # Limpieza de NaN antes de escalar y crear ventanas
    df = df.copy()
    df = df.dropna(subset=features + [target]).reset_index(drop=True)
    print(f'   🧹 Registros válidos (sin NaN): {df.shape[0]}')

    scaler_features = MinMaxScaler()
    df[features] = scaler_features.fit_transform(df[features])

    X_list, y_list = [], []
    pozos = df['Well_ID'].unique()

    for well in pozos:
        df_well = df[df['Well_ID'] == well].sort_values(by='Date')
        X_well = df_well[features].values
        y_well = df_well[target].values

        if len(X_well) > tamano_ventana:
            for i in range(len(X_well) - tamano_ventana):
                X_list.append(X_well[i: i + tamano_ventana])
                y_list.append(y_well[i + tamano_ventana])

    X_final = np.array(X_list)
    y_final = np.array(y_list)

    print(f'✅ Tensor X: {X_final.shape} | Vector y: {y_final.shape}')
    print(f'   ↳ {X_final.shape[0]} bloques de {X_final.shape[1]} días · {X_final.shape[2]} variables')
    return X_final, y_final, scaler_features


# Encadenado directo desde df_modelo
X, y, scaler = crear_ventanas_3d(df_modelo, tamano_ventana=30)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'\n📊 Entrenamiento: {X_train.shape[0]} muestras | Validación: {X_val.shape[0]} muestras')

In [ ]:
# ================================================================
# CELDA 6 — GRADIENT BOOSTING (Modelo comparativo)
# ================================================================
print('🧠 Entrenando Gradient Boosting Regressor...')

X_flat_train = X_train.reshape(X_train.shape[0], -1)
X_flat_val = X_val.reshape(X_val.shape[0], -1)

# Eliminar filas con NaN para evitar error en GradientBoosting
mask_train = ~np.isnan(X_flat_train).any(axis=1) & ~np.isnan(y_train)
mask_val = ~np.isnan(X_flat_val).any(axis=1) & ~np.isnan(y_val)
X_flat_train = X_flat_train[mask_train]
y_train_clean = y_train[mask_train]
X_flat_val = X_flat_val[mask_val]
y_val_clean = y_val[mask_val]
print(f'   Muestras train sin NaN: {X_flat_train.shape[0]} | val sin NaN: {X_flat_val.shape[0]}')

gb_model = GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42)
gb_model.fit(X_flat_train, y_train_clean)

pred_gb = gb_model.predict(X_flat_val)
mae_gb = mean_absolute_error(y_val_clean, pred_gb)
r2_gb = r2_score(y_val_clean, pred_gb)

print(f'\n🏆 Gradient Boosting:')
print(f'   MAE: {mae_gb:.2f} días')
print(f'   R²:  {r2_gb:.4f} ({r2_gb*100:.1f}% variabilidad explicada)')

plt.figure(figsize=(8, 6))
plt.scatter(y_val, pred_gb, color='teal', alpha=0.3, label='Predicciones GB')
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2, label='Línea Perfecta')
plt.title('Validación Gradient Boosting\nDías Reales vs Predichos')
plt.xlabel('Días Restantes Reales')
plt.ylabel('Días Restantes Predichos')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# CELDA 7 — LSTM: ENTRENAMIENTO + ALERTAS TEMPRANAS
# ================================================================
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential

print('🧠 Configurando arquitectura LSTM...')
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='linear')
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print('\n🚀 Entrenando la red LSTM...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# Curvas de aprendizaje
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_title('Pérdida (MSE) durante entrenamiento')
ax1.set_xlabel('Época')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['mae'], label='Train MAE')
ax2.plot(history.history['val_mae'], label='Val MAE')
ax2.set_title('Error Absoluto Medio (MAE)')
ax2.set_xlabel('Época')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Validación cruzada
print('\n🔮 Evaluando rendimiento LSTM...')
pred_lstm = model.predict(X_val).flatten()
mae_lstm = mean_absolute_error(y_val, pred_lstm)
r2_lstm = r2_score(y_val, pred_lstm)

print(f'\n🏆 LSTM:')
print(f'   MAE: {mae_lstm:.2f} días')
print(f'   R²:  {r2_lstm:.4f} ({r2_lstm*100:.1f}% variabilidad explicada)')

print(f'\n📊 Comparativa final:')
print(f'   Gradient Boosting → MAE: {mae_gb:.2f} | R²: {r2_gb:.4f}')
print(f'   LSTM              → MAE: {mae_lstm:.2f} | R²: {r2_lstm:.4f}')

plt.figure(figsize=(9, 7))
plt.scatter(y_val, pred_lstm, color='darkcyan', alpha=0.3, label='Predicciones LSTM')
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2, label='Línea Perfecta')
plt.title('Validación LSTM\nDías Reales vs Días Predichos con Memoria')
plt.xlabel('Días Restantes Reales')
plt.ylabel('Días Restantes Predichos (LSTM)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------
# ALERTAS TEMPRANAS POR POZO
# ----------------------------------------------------------------
print('\n📊 Generando gráficas de alerta temprana por pozo...')
features = ['Oil_rate_bbl_d', 'Water_rate_bbl_d', 'Gas_rate_mscf_d']
tamano_ventana = 30
pozos = df_modelo['Well_ID'].unique()

df_norm = df_modelo.copy()
df_norm[features] = scaler.transform(df_norm[features])

for well_id in pozos:
    df_pozo = df_norm[df_norm['Well_ID'] == well_id].copy()
    df_pozo['Date'] = pd.to_datetime(df_pozo['Date'])
    df_pozo = df_pozo.sort_values(by='Date').reset_index(drop=True)

    if len(df_pozo) <= tamano_ventana:
        continue

    X_pozo = np.array([df_pozo[features].values[i: i + tamano_ventana] for i in range(len(df_pozo) - tamano_ventana)])
    y_real = df_pozo['Dias_Restantes'].values[tamano_ventana:]
    fechas = df_pozo['Date'].values[tamano_ventana:]
    predicciones = model.predict(X_pozo, verbose=0).flatten()

    plt.figure(figsize=(11, 5))
    plt.plot(fechas, y_real, label='Días Reales (Histórico)', color='black', lw=2)
    plt.plot(fechas, predicciones, label='Predicción LSTM', color='crimson', linestyle='--', lw=1.8)
    plt.axhline(y=90, color='orange', linestyle=':', lw=2, label='Umbral de Alerta (90 Días)')
    plt.title(f'Alerta Temprana — Pozo: {well_id}', fontsize=12, fontweight='bold')
    plt.xlabel('Línea de Tiempo')
    plt.ylabel('Días Restantes para Colapso')
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper right', fontsize=9)
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.show()
    print(f'✔️ Alerta generada para pozo: {well_id}')

In [ ]:
# ================================================================
# CELDA 8 — EXPORTAR PREDICCIONES (Opcional)
# Guarda un Excel con las predicciones para Power BI u otros usos
# ================================================================
print('💾 Exportando predicciones del campo completo...')

data_exportar = []

for well_id in pozos:
    df_pozo = df_norm[df_norm['Well_ID'] == well_id].copy()
    df_pozo['Date'] = pd.to_datetime(df_pozo['Date'])
    df_pozo = df_pozo.sort_values(by='Date').reset_index(drop=True)

    if len(df_pozo) <= tamano_ventana:
        continue

    X_pozo = np.array([df_pozo[features].values[i: i + tamano_ventana] for i in range(len(df_pozo) - tamano_ventana)])
    predicciones = model.predict(X_pozo, verbose=0).flatten()

    df_recortado = df_modelo[df_modelo['Well_ID'] == well_id].iloc[tamano_ventana:].copy()
    df_recortado['Prediccion_LSTM_Dias'] = predicciones
    data_exportar.append(df_recortado)

df_exportar = pd.concat(data_exportar, ignore_index=True)
df_exportar.to_excel('Predicciones_Campo_Completo.xlsx', index=False)
print(f'🎯 Archivo exportado: Predicciones_Campo_Completo.xlsx ({df_exportar.shape[0]} registros)')